In [4]:
import os

print(os.listdir('/content'))

['.config', 'signatures1', 'Scan_16042026093714_SIG.zip', 'Xerox Scan_11052026123541_SIG.zip', 'sample_data']


In [5]:
import zipfile, os, glob

print(glob.glob('/content/*.zip'))

for zip_path in glob.glob('/content/*.zip'):
    print("Décompression de :", zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/signatures')

print(os.listdir('/content/signatures')[:10])

['/content/Scan_16042026093714_SIG.zip', '/content/Xerox Scan_11052026123541_SIG.zip']
Décompression de : /content/Scan_16042026093714_SIG.zip
Décompression de : /content/Xerox Scan_11052026123541_SIG.zip
['48271', '62492', 'Scan_16042026093714_SIG', '62276', '62370', '62496', '63838']


In [6]:
import os

print(len(os.listdir('/content/signatures1')))
print(os.listdir('/content/signatures1')[:5])

6
['48271', '62492', '62276', '62370', '62496']


In [8]:
import os
import cv2
from skimage.feature import hog

def build_signature_database(path='/content/signatures1'):

    database = {}

    for student_id in os.listdir(path):

        student_folder = os.path.join(path, student_id)

        if os.path.isdir(student_folder):

            features_list = []

            for img_name in os.listdir(student_folder):

                img_path = os.path.join(student_folder, img_name)

                img = cv2.imread(img_path, 0)

                if img is not None:

                    img = cv2.resize(img, (128, 64))

                    features = hog(img)

                    features_list.append(features)

            database[student_id] = features_list

    return database

In [9]:
db = build_signature_database()

print(db.keys())

dict_keys(['48271', '62492', '62276', '62370', '62496', '63838'])


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def authenticate_signature(test_features, database):

    best_score = -1
    best_id = None

    for student_id, features_list in database.items():

        for features in features_list:

            score = cosine_similarity(
                [test_features],
                [features]
            )[0][0]

            if score > best_score:

                best_score = score
                best_id = student_id

    return best_id, best_score

In [11]:
test_id = '63838'

test_features = db[test_id][0]

result_id, score = authenticate_signature(test_features, db)

print("ID trouvé :", result_id)
print("Score :", score)

ID trouvé : 63838
Score : 0.9999999999999996


In [12]:
for student_id in db.keys():
    test_features = db[student_id][0]
    result_id, score = authenticate_signature(test_features, db)
    print(student_id, "=> trouvé :", result_id, "| score :", score)

48271 => trouvé : 48271 | score : 0.9999999999999991
62492 => trouvé : 62492 | score : 1.0000000000000022
62276 => trouvé : 62276 | score : 1.0
62370 => trouvé : 62370 | score : 0.9999999999999987
62496 => trouvé : 62496 | score : 0.9999999999999986
63838 => trouvé : 63838 | score : 0.9999999999999996


In [13]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_student_id_from_grid(grid_img_path):
    """
    Lecture simple d'un ID étudiant depuis une grille cochée.
    Hypothèse : 5 colonnes = 5 chiffres de l'ID
                10 lignes = chiffres 0 à 9
    """

    img = cv2.imread(grid_img_path, 0)

    if img is None:
        print("Image introuvable")
        return ""

    # binarisation
    _, thresh = cv2.threshold(img, 180, 255, cv2.THRESH_BINARY_INV)

    h, w = thresh.shape

    nb_rows = 10
    nb_cols = 5

    cell_h = h // nb_rows
    cell_w = w // nb_cols

    student_id = ""

    for col in range(nb_cols):
        max_pixels = 0
        selected_digit = ""

        for row in range(nb_rows):
            y1 = row * cell_h
            y2 = (row + 1) * cell_h
            x1 = col * cell_w
            x2 = (col + 1) * cell_w

            cell = thresh[y1:y2, x1:x2]

            black_pixels = cv2.countNonZero(cell)

            if black_pixels > max_pixels:
                max_pixels = black_pixels
                selected_digit = str(row)

        student_id += selected_digit

    return student_id